# Trabajo en clase — Q-Learning con FrozenLake

En CheeseWorld construimos el algoritmo desde cero. Ahora utilizaremos el mismo procedimiento en un ambiente estándar de **Gymnasium**.

## Objetivo

Durante la clase debes relacionar cada parte del código con los conceptos:

- estado \(s\);
- acción \(a\);
- recompensa \(r\);
- Q-table;
- exploración y explotación;
- TD target;
- TD error;
- política greedy.

Este notebook tiene **espacios para discutir y escribir conclusiones durante la clase**.


## 1. Imports y funciones de Q-Learning


In [1]:
import numpy as np
import gymnasium as gym
import random
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output, display

from IPython.display import HTML
from matplotlib import animation
import matplotlib.pyplot as plt


def play_episode(env, Q=None, random_policy=False, max_steps=100, seed=None):
    """Ejecuta un episodio y devuelve sus frames y recompensa total."""
    state, _ = env.reset(seed=seed)
    frames = [env.render()]
    total_reward = 0.0

    for _ in range(max_steps):
        if random_policy:
            action = env.action_space.sample()
        else:
            q_values = Q[state]
            max_q = np.max(q_values)

            # Desempate aleatorio entre acciones con el mismo Q.
            best_actions = np.flatnonzero(q_values == max_q)
            action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)

        frames.append(env.render())
        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return frames, total_reward


def frames_to_video(frames, interval=700):
    """Convierte una lista de frames RGB en una animación reproducible en Jupyter."""
    fig = plt.figure(figsize=(4, 4))
    plt.axis("off")

    image = plt.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


def initialize_q_table(state_space, action_space):
    return np.zeros((state_space, action_space))


def greedy_policy(Qtable, state):
    # Si hay empate entre varias acciones con el mismo Q, desempata al azar.
    max_q = np.max(Qtable[state])
    best_actions = np.flatnonzero(Qtable[state] == max_q)
    return int(np.random.choice(best_actions))


def epsilon_greedy_policy(Qtable, state, epsilon, env):
    if random.random() < epsilon:
        return env.action_space.sample()

    return greedy_policy(Qtable, state)


def train_q_learning(
    env,
    Qtable,
    n_episodes=5000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    max_steps=100,
    start_episode=0,
):
    rewards = []

    for episode in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        global_episode = start_episode + episode
        epsilon = min_epsilon + (
            max_epsilon - min_epsilon
        ) * np.exp(-decay_rate * global_episode)

        for _ in range(max_steps):
            action = epsilon_greedy_policy(
                Qtable, state, epsilon, env
            )

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            best_next_q = 0.0 if done else np.max(Qtable[next_state])

            td_target = reward + gamma * best_next_q
            td_error = td_target - Qtable[state, action]

            Qtable[state, action] += learning_rate * td_error

            state = next_state
            total_reward += reward

            if done:
                break

        rewards.append(total_reward)

    return Qtable, rewards


def evaluate_q_policy(env, Qtable, n_episodes=100, max_steps=100):
    episode_rewards = []

    for _ in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):
            action = greedy_policy(Qtable, state)

            next_state, reward, terminated, truncated, _ = env.step(action)

            state = next_state
            total_reward += reward

            if terminated or truncated:
                break

        episode_rewards.append(total_reward)

    return np.mean(episode_rewards), np.std(episode_rewards)


def show_frame(env, title=''):
    frame = env.render()
    plt.figure(figsize=(4, 4))
    plt.imshow(frame)
    plt.axis('off')
    plt.title(title)
    display(plt.gcf())
    plt.close()

In [2]:
# Semilla para que los resultados (y las respuestas escritas) sean reproducibles
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

### 💬 Antes de ejecutar

En CheeseWorld teníamos explícitamente una clase `Environment` y una clase `QLearningAgent`.

**Pregunta:** en este notebook, ¿qué papel cumple Gymnasium y dónde quedó representado el agente?

**Notas:**

- **Gymnasium es el ambiente.** Reemplaza a la clase `Environment` de CheeseWorld: define los estados (`observation_space`), las acciones (`action_space`), la dinámica de transición y la recompensa. Todo eso queda oculto detrás de `env.reset()` y `env.step(action)`.
- **El agente ya no es una clase**, sino la combinación de la **Q-table** (`Q`, un arreglo NumPy de estados × acciones) y las funciones que operan sobre ella: `epsilon_greedy_policy` (cómo decide), `train_q_learning` (cómo aprende con la regla de Q-Learning) y `evaluate_q_policy` (cómo actúa de forma greedy).
- La interfaz entre ambos es la misma de siempre: el agente envía una acción $a$ y el ambiente devuelve $(r, s')$ más las banderas `terminated`/`truncated`.


## 2. Crear FrozenLake


In [3]:
env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False,
    render_mode="rgb_array"
)

env.action_space.seed(SEED)
state, info = env.reset(seed=SEED)

print("Estado inicial:", state)
print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)

Estado inicial: 0
Número de estados: 16
Número de acciones: 4


En FrozenLake las acciones son:

| Acción | Código |
|---|---:|
| Left | 0 |
| Down | 1 |
| Right | 2 |
| Up | 3 |

Primero trabajaremos con `is_slippery=False`, es decir, con transiciones determinísticas.


### 💬 Actividad 1 — La Q-table

Antes de crearla:

1. ¿Cuántas filas debe tener la Q-table?
2. ¿Cuántas columnas?
3. ¿Qué representa una celda \(Q[s,a]\)?

**Respuesta / discusión:**

- **16 filas**, una por estado: el mapa 4×4 tiene 16 casillas (`env.observation_space.n = 16`).
- **4 columnas**, una por acción: Left, Down, Right, Up (`env.action_space.n = 4`).
- $Q[s,a]$ es la **estimación del retorno esperado (descontado)** que obtiene el agente si en el estado $s$ ejecuta la acción $a$ y de ahí en adelante sigue actuando de forma óptima. No es la recompensa inmediata, sino el valor a largo plazo de esa decisión.


In [4]:
state_space = env.observation_space.n
action_space = env.action_space.n

Q = initialize_q_table(state_space, action_space)

print("Q-table shape:", Q.shape)
Q

Q-table shape: (16, 4)


array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])

### 💬 Actividad 2 — Inicio del aprendizaje

Todos los valores son cero.

$$
Q(s,a)=0
$$

¿Esto significa que todas las acciones son malas, o que el agente todavía no sabe nada?

¿Qué ocurre si varias acciones tienen exactamente el mismo valor máximo?

**Notas:**

- Significa que **el agente todavía no sabe nada**: el cero es solo una inicialización neutra, no una evaluación. En FrozenLake además la única recompensa es $+1$ al llegar a la meta, así que los valores solo empiezan a crecer cuando alguna trayectoria llega a `G` y esa recompensa se propaga hacia atrás con $\gamma \max_{a'} Q(s',a')$.
- Si varias acciones empatan en el máximo, **cualquiera es igualmente "greedy"**. Por eso las funciones del notebook desempatan **al azar** (`np.flatnonzero(q_values == max_q)` + `np.random.choice`). Si siempre se tomara la primera (como hace `np.argmax`), el agente elegiría siempre `Left` al inicio y quedaría sesgado y atascado contra el borde.


## 3. Ejecutar una transición


In [5]:
state, _ = env.reset(seed=SEED)

epsilon = 1.0
action = epsilon_greedy_policy(Q, state, epsilon, env)

next_state, reward, terminated, truncated, _ = env.step(action)

print("state      =", state)
print("action     =", action)
print("reward     =", reward)
print("next_state =", next_state)
print("done       =", terminated or truncated)

state      = 0
action     = 0
reward     = 0
next_state = 0
done       = False


### 💬 Actividad 3 — Identificar la experiencia

Escribe la experiencia anterior como:

$$
(s,a,r,s')
$$

**Experiencia** (con `SEED = 42`, ver la salida de la celda anterior):

$$
(s,a,r,s') = (0,\ 0,\ 0,\ 0)
$$

El agente estaba en la casilla inicial $s=0$, eligió `Left` ($a=0$), chocó con el borde izquierdo, recibió $r=0$ y quedó en el mismo estado $s'=0$.

¿De cuál de esos cuatro elementos **no disponíamos directamente** en Value Iteration cuando hablábamos de experiencia real?

**Respuesta:** del **estado siguiente $s'$ observado** (y con él de la recompensa obtenida). En Value Iteration no se ejecuta nada: se usa el **modelo** $T(s,a,s')$ para sumar sobre *todos* los $s'$ posibles ponderados por su probabilidad. En Q-Learning no conocemos $T$; solo vemos **una muestra** $s'$ que el ambiente nos devuelve al actuar.


## 4. Del azar a una política aprendida

Vamos a observar **el mismo agente en tres momentos**. Primero no sabe nada y actúa al azar; luego veremos su política después de pocas experiencias; finalmente veremos la política después del entrenamiento completo.


In [6]:
# Guardaremos tres momentos del aprendizaje

# Momento 1: sin entrenamiento
Q_initial = initialize_q_table(
    env.observation_space.n,
    env.action_space.n
)

# Momento 2: poco entrenamiento
EARLY_EPISODES = 50
Q_early = Q_initial.copy()
Q_early, rewards_early = train_q_learning(
    env,
    Q_early,
    n_episodes=EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
)

# Momento 3: continuar hasta 10 000 episodios
TOTAL_EPISODES = 10000
Q_trained = Q_early.copy()
Q_trained, rewards_final = train_q_learning(
    env,
    Q_trained,
    n_episodes=TOTAL_EPISODES - EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    start_episode=EARLY_EPISODES,
)

Q = Q_trained
rewards = rewards_early + rewards_final

print('Snapshots guardados:')
print('Q_initial : 0 episodios')
print(f'Q_early   : {EARLY_EPISODES} episodios')
print(f'Q_trained : {TOTAL_EPISODES} episodios')


Snapshots guardados:
Q_initial : 0 episodios
Q_early   : 50 episodios
Q_trained : 10000 episodios


### Momento 1 — Sin entrenamiento: random walk
Todavía no usamos la Q-table para decidir. Cada acción se selecciona aleatoriamente. Observa cómo interactúa el agente con el mundo.

In [7]:
frames_random, reward_random = play_episode(
    env,
    Q_initial,
    random_policy=True,
    seed=7
)

print(f"Recompensa total: {reward_random}")
frames_to_video(frames_random, interval=700)


Recompensa total: 0.0


### Momento 2 — Después de pocas iteraciones

Ahora el agente usa de forma **greedy** lo que ha aprendido en `Q_early`. Todavía conoce poco del ambiente, así que su comportamiento puede ser incompleto o equivocarse.


In [8]:
frames_early, reward_early = play_episode(
    env,
    Q_early,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_early}")
frames_to_video(frames_early, interval=700)


Recompensa total: 1.0


### Momento 3 — Agente entrenado

Finalmente usamos `Q_trained`. Ya no exploramos: en cada estado el agente selecciona una de las acciones con mayor valor $Q(s,a)$.


In [9]:
frames_trained, reward_trained = play_episode(
    env,
    Q_trained,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_trained}")
frames_to_video(frames_trained, interval=700)


Recompensa total: 1.0


### 💬 Actividad — ¿Qué cambió?

Compara las tres ejecuciones. El ambiente, los estados y las acciones son los mismos. **¿Qué cambió internamente en el agente para que su comportamiento mejore?**

Observa `Q_initial`, `Q_early` y `Q_trained` y relaciona sus valores con las acciones que viste ejecutar.

**Respuesta:**

Lo único que cambió es la **Q-table**. El ambiente, el mapa y las acciones disponibles son idénticos en las tres ejecuciones.

- **`Q_initial`**: todo es cero y además usamos una política aleatoria. El agente deambula sin dirección y en esta ejecución termina sin recompensa (0.0): cae en un hoyo o se le acaban los pasos.
- **`Q_early` (50 episodios)**: casi toda la tabla sigue en cero. Con $\epsilon$ cercano a 1 casi todas las acciones fueron exploratorias, y la recompensa $+1$ apenas empezó a propagarse a los estados más cercanos a la meta (por ejemplo, `Q_early[14] = [0, 0, 0.7, 0]`: tras un solo éxito ya "sabe" que `Right` es buena ahí, $\alpha \cdot 1 = 0.7$). En los estados lejanos la política greedy todavía desempata al azar, así que su éxito en esta corrida (recompensa 1.0) depende en buena parte de la suerte. Con otra semilla puede fallar.
- **`Q_trained` (10 000 episodios)**: la recompensa se propagó desde la meta hacia atrás. Los valores forman un "gradiente" del tipo $\gamma^{k}$ (1.0 junto a la meta, 0.95, 0.9025, …), de modo que en cada estado `argmax` apunta a la acción que acerca a la meta por el camino más corto sin pasar por hoyos. Por eso el agente llega a `G` de forma consistente (recompensa 1.0 y media de evaluación 1.000).


### 💬 Actividad 4 — Leer una fila de Q

Selecciona un estado $s$ y observa:

$$
Q(s,0), Q(s,1), Q(s,2), Q(s,3)
$$

**Estado seleccionado:** $s = 14$ (fila 3, columna 2: la casilla inmediatamente a la izquierda de la meta `G`). Los valores se imprimen en la celda siguiente.

**Valores Q** (`Q_trained[14]`):

- Left: 0.9025
- Down: 0.9500
- Right: 1.0000
- Up: 0.9025

¿Cuál acción seleccionaría:

$$
\arg\max_a Q(s,a)
$$

?

**Right** ($a=2$), con $Q=1.0$.

**Interpretación:**

> Ir a la derecha lleva directamente a la meta y da $r=+1$, por eso $Q(14,\text{Right}) = 1$. `Down` choca con el borde inferior y deja al agente en el mismo estado; desde ahí todavía necesita un paso más, así que vale $\gamma \cdot 1 = 0.95$. `Left` y `Up` lo alejan un paso más ($\gamma^2 = 0.9025$). Los valores coinciden exactamente con potencias de $\gamma=0.95$ según cuántos pasos faltan para la meta. El ambiente es determinístico y la tabla convergió.


In [10]:
# Lectura de una fila de la Q-table entrenada (estado 14: celda a la izquierda de la meta)
s_sel = 14
acciones = ["Left", "Down", "Right", "Up"]
for a, nombre in enumerate(acciones):
    print(f"{nombre:5s}: {Q_trained[s_sel, a]:.4f}")
print("argmax_a Q(s,a) =", acciones[int(np.argmax(Q_trained[s_sel]))])
print("\nQ_initial[14] =", Q_initial[s_sel])
print("Q_early[14]   =", np.round(Q_early[s_sel], 4))

Left : 0.9025
Down : 0.9500
Right: 1.0000
Up   : 0.9025
argmax_a Q(s,a) = Right

Q_initial[14] = [0. 0. 0. 0.]
Q_early[14]   = [0.  0.  0.7 0. ]


## 5. Evaluar la política aprendida


In [11]:
mean_reward, std_reward = evaluate_q_policy(
    env,
    Q_trained,
    n_episodes=100
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")


Mean reward: 1.000
Std reward : 0.000


### 💬 Actividad 5 — Exploration vs. exploitation

Durante entrenamiento usamos $\epsilon$-greedy.

Durante evaluación usamos:

$$
a=\arg\max_aQ(s,a)
$$

¿Por qué **no exploramos** durante la evaluación?

**Conclusión:**

> La exploración sirve para **aprender**: obliga al agente a probar acciones cuyo valor todavía no conoce bien. En la evaluación ya no se actualiza la Q-table; lo que queremos medir es **qué tan buena es la política aprendida**. Si siguiéramos tomando acciones aleatorias con probabilidad $\epsilon$, la métrica mezclaría el desempeño de la política con el ruido de la exploración y la subestimaría (en FrozenLake un solo paso al azar puede tirar al agente a un hoyo). Por eso evaluamos con la política greedy $\pi(s)=\arg\max_a Q(s,a)$: en el caso determinístico obtiene media 1.000 y desviación 0.000.


## 6. Experimento: FrozenLake estocástico


In [12]:
slippery_env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=True,
    render_mode="rgb_array"
)

slippery_env.action_space.seed(SEED)
slippery_env.reset(seed=SEED)

Q_slippery = initialize_q_table(
    slippery_env.observation_space.n,
    slippery_env.action_space.n
)

Q_slippery, rewards_slippery = train_q_learning(
    slippery_env,
    Q_slippery,
    n_episodes=20000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.0005,
    max_steps=100
)

mean_reward, std_reward = evaluate_q_policy(
    slippery_env,
    Q_slippery,
    n_episodes=500
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")

Mean reward: 0.702
Std reward : 0.457


### 💬 Actividad 6 — Determinístico vs. estocástico

Compara:

- `is_slippery=False`
- `is_slippery=True`

¿Qué cambia en el **ambiente**?

¿Qué cambia en la **ecuación de Q-Learning**?

**Discusión:**

- **Ambiente:** con `is_slippery=True` las transiciones son **estocásticas**: la acción elegida solo se ejecuta con probabilidad 1/3, y con 1/3 cada una el agente se desliza a una de las dos direcciones perpendiculares. El mismo par $(s,a)$ puede producir distintos $s'$, así que ni la política óptima garantiza llegar a la meta. Por eso la recompensa media cae de **1.000** (determinístico) a **≈0.70** con desviación ≈0.46 (estocástico), y hubo que entrenar más episodios (20 000) con un decaimiento de $\epsilon$ más lento.
- **Algoritmo:** **la ecuación de Q-Learning no cambia**:
  $Q(s,a)\leftarrow Q(s,a)+\alpha\,[\,r+\gamma\max_{a'}Q(s',a')-Q(s,a)\,]$.
  Lo que cambia es su interpretación: cada actualización usa **una muestra** de $s'$, y al promediar muchas muestras (gracias a $\alpha$) $Q$ aproxima la esperanza $\sum_{s'}T(s,a,s')[\dots]$ sin conocer $T$. Esa es la ventaja de ser *model-free*. En la práctica, con transiciones ruidosas conviene un $\alpha$ más pequeño: con $\alpha=0.7$ los valores oscilan mucho de una actualización a otra.


# Cierre de clase

Completa antes de terminar:

**1. ¿Qué almacena $Q(s,a)$?**

> El retorno esperado (suma descontada de recompensas futuras) al ejecutar la acción $a$ en el estado $s$ y seguir después la mejor política. Es el "valor a largo plazo" de esa decisión, no la recompensa inmediata.

**2. ¿De dónde sale $\max_{a'}Q(s',a')$?**

> De la **propia Q-table**, evaluada en el estado siguiente $s'$ que el ambiente devolvió. Es la estimación actual del mejor valor alcanzable desde $s'$. Usar una estimación para mejorar otra estimación es lo que se llama *bootstrapping*. Que sea un `max` (y no la acción que realmente se tomará) hace a Q-Learning un método **off-policy**.

**3. ¿Por qué necesitamos $\epsilon$-greedy?**

> Para equilibrar **exploración** y **explotación**. Si el agente siempre fuera greedy, se quedaría con la primera acción que le diera algo de valor y nunca descubriría caminos mejores (o, con todo en cero, nunca llegaría a la meta). Con probabilidad $\epsilon$ prueba acciones al azar, y con $1-\epsilon$ aprovecha lo aprendido. Al hacer decaer $\epsilon$ explora mucho al inicio y explota al final.

**4. ¿Por qué Q-Learning es model-free?**

> Porque **no necesita conocer** la función de transición $T(s,a,s')$ ni la de recompensa $R$. Aprende solo con experiencias muestreadas $(s,a,r,s')$ obtenidas al interactuar con el ambiente, a diferencia de Value/Policy Iteration, que requieren el modelo completo para calcular $\sum_{s'}T(s,a,s')V(s')$.
